In [0]:
%pip install prophet

In [0]:
catalog = "main"
schema = "ray_gtm_examples"
table = "data_synthetic_timeseries_1000_groups"
write_table = "prophet_binaries"
label="y"

In [0]:
query = f"""
SELECT *
FROM {catalog}.{schema}.{table}
WHERE group_name = 'group_01'
"""
grouped_data = spark.sql(query).toPandas()

In [0]:
from prophet import Prophet

horizon=14


# fit the model and generate forecasts
m = Prophet(daily_seasonality=True)
m.fit(grouped_data)



In [0]:
import pickle
from pyspark.sql import Row
from datetime import datetime

# Serialize the Prophet model to a binary string
model_binary = pickle.dumps(m)

# Save all models to Delta
model_df = spark.createDataFrame([Row(group_name='group_01',
                                      model_binary=model_binary,
                                      algorithm='prophet',
                                      creation_time=datetime.now())])
# model_df.write.mode("overwrite").saveAsTable(write_table)
model_df.display()


In [0]:
loaded = pickle.loads(model_binary)
future = loaded.make_future_dataframe(periods=horizon)
forecast = loaded.predict(future)


In [0]:
forecast

In [0]:
import pandas as pd

to_write = pd.DataFrame({'group_name': ['group_01'],
                        'model_binary': [model_binary],
                        'algorithm': ['prophet'],
                        'creation_time': [str(datetime.now())]})

In [0]:
to_write